# 🧹 Somali News Dataset — Cleaning Pipeline v2
**Thesis:** Automatic Somali Headline Generation Using Transformer-Based NLP Models

This notebook was built after a deep analysis of all 17 raw CSV files.
Every cleaning rule below is backed by an actual pattern found in the real data.

| Cell | Section |
|------|---------|
| 0    | Install dependencies |
| 1    | Imports & configuration |
| 2    | Load all raw files |
| 3    | Raw data inspection report |
| 4    | All cleaning functions (run once) |
| 5    | Remove nav/TOS/boilerplate articles |
| 6    | Clean headlines |
| 7    | Clean article bodies |
| 8    | Handle missing values |
| 9    | Deduplicate |
| 10   | Suspicious article detection |
| 11   | Category & source validation |
| 12   | Save cleaned files |
| 13   | Build model-ready dataset |
| 14   | Final summary report |

## Cell 0 — Install Dependencies

In [1]:
# Run this cell only on first use.
# After it finishes, restart the kernel, then run all other cells.
import subprocess, sys
for pkg in ['pandas', 'numpy', 'tqdm', 'tabulate']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True)
    print('✔' if r.returncode == 0 else '✘', pkg)
print('\nDone. Restart the kernel now if this was your first run.')

✔ pandas
✔ numpy
✔ tqdm
✔ tabulate

Done. Restart the kernel now if this was your first run.


## Cell 1 — Imports & Configuration

In [2]:
"""
CELL 1 — IMPORTS & CONFIGURATION
=================================
All project-wide settings live here so you only need to change one place.
- Paths: where raw files are, where cleaned files go
- Quality thresholds: minimum headline/body length
- Valid categories: controls what goes in the final dataset
"""

import pandas as pd
import numpy as np
import re
import os
import hashlib
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
tqdm.pandas(desc='Processing')

# ─── Folder paths ─────────────────────────────────────────────────────────────
# Put all your raw CSVs inside data/raw/ before running Cell 2
BASE_DIR    = Path('.')
RAW_DIR     = BASE_DIR / 'data' / 'raw'
CLEANED_DIR = BASE_DIR / 'data' / 'cleaned'
REVIEW_DIR  = BASE_DIR / 'data' / 'review'
REPORTS_DIR = BASE_DIR / 'data' / 'reports'

for d in [RAW_DIR, CLEANED_DIR, REVIEW_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Valid categories ─────────────────────────────────────────────────────────
VALID_CATEGORIES = ['siyaasad', 'amni', 'caalamka', 'ciyaaro']

# ─── Quality thresholds (based on data analysis) ──────────────────────────────
# These values were chosen after inspecting the actual shortest/longest articles
MIN_HEADLINE_CHARS = 15    # anything shorter is likely a nav item or stub
MAX_HEADLINE_CHARS = 300   # anything longer is likely a scraped paragraph
MIN_BODY_CHARS     = 150   # articles shorter than this have no useful content
MIN_BODY_WORDS     = 30    # second check: word count

TODAY = datetime.now().strftime('%Y-%m-%d')

print('✅ Configuration loaded.')
print(f'   Raw data folder  : {RAW_DIR.resolve()}')
print(f'   Cleaned output   : {CLEANED_DIR.resolve()}')
print(f'   Valid categories : {VALID_CATEGORIES}')

✅ Configuration loaded.
   Raw data folder  : E:\somali_cleaner\data\raw
   Cleaned output   : E:\somali_cleaner\data\cleaned
   Valid categories : ['siyaasad', 'amni', 'caalamka', 'ciyaaro']


In [24]:
"""
CELL 2 — LOAD + COMBINE ALL RAW FILES
=====================================
This cell:
- Reads all raw CSV / Excel files from data/raw/
- Combines them into one DataFrame called raw_df
- Standardizes columns for the cleaning notebook:
    title   → headline
    content → body
- Adds missing site/category from filename when needed
- Saves one combined raw file:
    data/combined_raw_data.csv
"""

import pandas as pd
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR = Path(".")
RAW_DIR = BASE_DIR / "data" / "raw"
OUTPUT_PATH = BASE_DIR / "data" / "combined_raw_data.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Find all raw files ───────────────────────────────────────────────────────
raw_files = []

for pattern in ["*.csv", "*.xlsx", "*.xls"]:
    raw_files.extend(RAW_DIR.glob(pattern))

# Avoid reading old combined output if it was accidentally placed in raw folder
raw_files = [
    fp for fp in sorted(raw_files)
    if "combined_raw_data" not in fp.stem.lower()
]

if not raw_files:
    raise FileNotFoundError(f"❌ No CSV or Excel files found in: {RAW_DIR.resolve()}")

print(f"✅ Found {len(raw_files)} raw files in: {RAW_DIR.resolve()}")
print("─" * 80)

frames = []

# ── Helper: infer site/category from filename ────────────────────────────────
def infer_site_category(file_stem: str):
    """
    Examples:
      bbc_somali__ciyaaro      → site=bbc_somali, category=ciyaaro
      caasimada__siyaasad      → site=caasimada, category=siyaasad
      goobjoog__caalamka       → site=goobjoog, category=caalamka
      laacibnet__ciyaaro       → site=laacibnet, category=ciyaaro
    """
    file_stem = file_stem.strip().lower()

    if "__" in file_stem:
        site, category = file_stem.split("__", 1)
    else:
        parts = file_stem.split("_")
        category = parts[-1] if len(parts) > 1 else ""
        site = "_".join(parts[:-1]) if len(parts) > 1 else file_stem

    return site.strip(), category.strip()


# ── Read and combine files ───────────────────────────────────────────────────
for fp in raw_files:
    try:
        print(f"📄 Reading: {fp.name}")

        if fp.suffix.lower() == ".csv":
            df = pd.read_csv(fp, dtype=str, encoding="utf-8-sig")
        elif fp.suffix.lower() in [".xlsx", ".xls"]:
            df = pd.read_excel(fp, dtype=str)
        else:
            print(f"   ⚠ Skipped unsupported file: {fp.name}")
            continue

        df = df.dropna(how="all").copy()

        # Normalize column names
        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        # Rename scraper columns to notebook columns
        rename_map = {
            "title": "headline",
            "content": "body",
            "article_title": "headline",
            "article_body": "body",
            "link": "url",
        }

        df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

        # Infer site/category from filename
        site_from_file, category_from_file = infer_site_category(fp.stem)

        # Add required columns if missing
        if "site" not in df.columns:
            df["site"] = site_from_file

        if "category" not in df.columns:
            df["category"] = category_from_file

        if "url" not in df.columns:
            df["url"] = ""

        if "headline" not in df.columns:
            df["headline"] = ""

        if "body" not in df.columns:
            df["body"] = ""

        if "scraped_at" not in df.columns:
            df["scraped_at"] = ""

        # Add trace column
        df["source_file"] = fp.name

        # Keep all columns, but make sure important ones exist
        for col in ["site", "category", "url", "headline", "body", "scraped_at", "source_file"]:
            df[col] = df[col].astype(str).fillna("").str.strip()

        frames.append(df)

        print(f"   ✅ Loaded {len(df):,} rows")

    except Exception as e:
        print(f"   ❌ Failed: {fp.name}")
        print(f"      Error: {e}")

if not frames:
    raise ValueError("❌ No files were successfully loaded.")

# ── Combine all raw data ─────────────────────────────────────────────────────
raw_df = pd.concat(frames, ignore_index=True)

# Remove fully empty rows after combining
raw_df = raw_df.dropna(how="all").copy()

# Replace bad string values
raw_df = raw_df.replace({
    "nan": "",
    "None": "",
    "NONE": "",
    "null": "",
    "NULL": "",
})

# ── Save combined raw file ───────────────────────────────────────────────────
raw_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# Also create these variables because the rest of your notebook uses them
TOTAL_RAW = len(raw_df)

print("\n" + "═" * 80)
print("✅ ALL RAW FILES COMBINED SUCCESSFULLY")
print("═" * 80)
print(f"📊 Total rows combined : {TOTAL_RAW:,}")
print(f"📊 Total files loaded  : {len(frames):,}")
print(f"📁 Saved file          : {OUTPUT_PATH.resolve()}")

print("\n📌 Rows by source file:")
display(
    raw_df["source_file"]
    .value_counts()
    .rename_axis("source_file")
    .reset_index(name="rows")
)

print("\n📌 Rows by category:")
display(
    raw_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

print("\n📌 Preview:")
display(raw_df.head())

✅ Found 16 raw files in: E:\somali_cleaner\data\raw
────────────────────────────────────────────────────────────────────────────────
📄 Reading: bbc_somali__ciyaaro.csv
   ✅ Loaded 245 rows
📄 Reading: caasimada__caalamka.csv
   ✅ Loaded 4,049 rows
📄 Reading: caasimada__siyaasad.csv
   ✅ Loaded 5,316 rows
📄 Reading: garoweonline__ciyaaro.csv
   ✅ Loaded 11 rows
📄 Reading: goobjoog__amni.csv
   ✅ Loaded 6,595 rows
📄 Reading: goobjoog__caalamka.csv
   ✅ Loaded 1,519 rows
📄 Reading: goobjoog__siyaasad.csv
   ✅ Loaded 4,746 rows
📄 Reading: kooxda__ciyaaro.csv
   ✅ Loaded 7,193 rows
📄 Reading: kooxdamanta__ciyaaro.csv
   ✅ Loaded 8 rows
📄 Reading: kubadlive__ciyaaro.csv
   ✅ Loaded 114 rows
📄 Reading: laacibnet__ciyaaro.csv
   ✅ Loaded 1,094 rows
📄 Reading: mustaqbalmedia__caalamka.csv
   ✅ Loaded 4,548 rows
📄 Reading: mustaqbalmedia__ciyaaro.csv
   ✅ Loaded 92 rows
📄 Reading: puntlandpost__ciyaaro.csv
   ✅ Loaded 226 rows
📄 Reading: sonna__ciyaaro.csv
   ✅ Loaded 26 rows
📄 Reading: wararka24

,source_file,rows
0,kooxda__ciyaaro.csv,7193
1,goobjoog__amni.csv,6595
2,caasimada__siyaasad.csv,5316
3,goobjoog__siyaasad.csv,4746
4,mustaqbalmedia__caalamka.csv,4548
5,caasimada__caalamka.csv,4049
6,goobjoog__caalamka.csv,1519
7,laacibnet__ciyaaro.csv,1094
8,bbc_somali__ciyaaro.csv,245
9,puntlandpost__ciyaaro.csv,226



📌 Rows by category:


,category,rows
0,caalamka,10116
1,siyaasad,10062
2,ciyaaro,9024
3,amni,6595



📌 Preview:


,id,site,category,url,headline,body,scraped_at,word_count,source_file
0,bbc_somali__ciyaaro__000001,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/clyp28rnzw0o,Maxay dalalku ka faa'idaan marka ay u soo baxaan ka qeybgalka Koobka Adduunka?,"Xigashada Sawirka, Getty Images Dalalka si guul leh ugu soo baxa Koobka Adduunka ee FIFA waxay helaan sharaf weyn oo...",2026-05-19 20:30:29,656,bbc_somali__ciyaaro.csv
1,bbc_somali__ciyaaro__000002,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cd9plkkj1pdo,Maxaa ka jira in ciyaartooyda kooxdii caanka ahayd ee Real Madrid mid mid isu dilayaan?,"Xigashada Sawirka, Getty Images Kooxda Real Madrid ayaa toddobaadkan gashay xaalad qalalaase weyn ah, xilli ay ahayd...",2026-05-19 20:30:33,809,bbc_somali__ciyaaro.csv
2,bbc_somali__ciyaaro__000003,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cjdplv01z52o,Ciyaaryahanada ugu sarreeya Afrika ee aan ka qayb galayn Koobka Adduunka 2026,"Xigashada Sawirka, Getty Images Iyada oo ay soo dhowaatay isku-aadka Koobka Adduunka, dalal badan, gaar ahaan kuwa k...",2026-05-19 20:30:36,616,bbc_somali__ciyaaro.csv
3,bbc_somali__ciyaaro__000004,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/c332veek778o,Maxaa ugu wacan Arsenal inaysan weligeed ku guulaysan Champions League,"Xigashada Sawirka, Getty Images Arsenal waligeed kuma guuleysan Champions League iyo Europa League. Hadda waxay soo ...",2026-05-19 20:30:39,848,bbc_somali__ciyaaro.csv
4,bbc_somali__ciyaaro__000005,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cly07pjzdm0o,Dhamaan xogta iyo macluumaadka aad u baahan tahay in aad ka ogaato koobka kubadda cagta adduunka ee 2026,"Xigashada Sawirka, Getty Images Iyadoo dhamaan kooxaha ka qayb galaya koobka adduunka hadda la og yahay, indhaha aya...",2026-05-19 20:30:42,1731,bbc_somali__ciyaaro.csv


## Cell 2 — Load All Raw Files

In [3]:
"""
LOAD ALL RAW FILES
============================
Scans data/raw/ for all CSV files and loads them into one combined DataFrame.

Key actions:
- Reads every CSV with dtype=str to prevent pandas from misreading Somali text
- Renames 'title' → 'headline' and 'content' → 'body' for clarity
- Adds a 'source_file' column so we can always trace which file a row came from
- Skips files that are missing required columns

"""

REQUIRED_COLS = ['site', 'category', 'url', 'title', 'content']

raw_files = sorted(RAW_DIR.glob('*.csv'))

if not raw_files:
    print('⚠  No CSV files found in data/raw/')
    print('   Copy your scraped CSV files there and re-run this cell.')
else:
    frames = []
    print(f'  {"FILE":<45} {"ROWS":>7}')
    print('  ' + '─' * 55)

    for fp in raw_files:
        try:
            df = pd.read_csv(fp, dtype=str)   # dtype=str preserves Somali text

            # Check required columns exist
            missing = [c for c in REQUIRED_COLS if c not in df.columns]
            if missing:
                print(f'  ⚠  {fp.name}: missing columns {missing} — skipped')
                continue

            # Rename for clarity throughout the rest of the notebook
            df = df.rename(columns={'title': 'headline', 'content': 'body'})

            # Track which file each row came from
            df['source_file'] = fp.name

            frames.append(df)
            print(f'  ✔  {fp.name:<45} {len(df):>7,}')

        except Exception as e:
            print(f'  ✘  {fp.name}: {e}')

    # Combine all into one DataFrame
    raw_df = pd.concat(frames, ignore_index=True)
    TOTAL_RAW = len(raw_df)

    print('  ' + '─' * 55)
    print(f'  Total: {TOTAL_RAW:,} rows across {len(frames)} files')

  FILE                                             ROWS
  ───────────────────────────────────────────────────────
  ✔  bbc_somali__ciyaaro.csv                           245
  ✔  caasimada__caalamka.csv                         4,049
  ✔  caasimada__siyaasad.csv                         5,316
  ✔  garoweonline__ciyaaro.csv                          11
  ✔  goobjoog__amni.csv                              6,595
  ✔  goobjoog__caalamka.csv                          1,519
  ✔  goobjoog__siyaasad.csv                          4,746
  ✔  kooxda__ciyaaro.csv                             7,193
  ✔  kooxdamanta__ciyaaro.csv                            8
  ✔  kubadlive__ciyaaro.csv                            114
  ✔  laacibnet__ciyaaro.csv                          1,094
  ✔  mustaqbalmedia__caalamka.csv                    4,548
  ✔  mustaqbalmedia__ciyaaro.csv                        92
  ✔  puntlandpost__ciyaaro.csv                         226
  ✔  sonna__ciyaaro.csv                                 26
 

In [4]:
"""
DIAGNOSTIC — Find English articles in the dataset
Run this once to understand the problem before cleaning.
"""
import pandas as pd
from pathlib import Path

RAW_DIR = Path('data/raw')

# Common Somali function words — appear in almost every Somali sentence
SOMALI_MARKERS = {
    'ayaa', 'ayuu', 'ayey', 'waxaa', 'waxuu', 'waxay',
    'oo', 'iyo', 'ee', 'ku', 'ka', 'la', 'uu', 'ay',
    'in', 'waa', 'ah', 'ama', 'haddii', 'sidoo', 'kale',
    'waxa', 'lagu', 'looga', 'ugu', 'kaga', 'iska',
    'dalka', 'magaalada', 'dowladda', 'shacabka',
}

# Common English function words
ENGLISH_MARKERS = {
    'the', 'and', 'is', 'in', 'of', 'to', 'a', 'that',
    'it', 'for', 'with', 'on', 'are', 'was', 'has',
    'have', 'he', 'she', 'they', 'this', 'an', 'at',
    'by', 'from', 'or', 'be', 'as', 'not', 'but', 'his',
}

def detect_language(text: str) -> tuple:
    """
    Returns (language, somali_ratio, english_ratio).
    Language is 'somali', 'english', or 'mixed'.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ('unknown', 0, 0)

    words = text.lower().split()
    if not words:
        return ('unknown', 0, 0)

    so_count = sum(1 for w in words if w in SOMALI_MARKERS)
    en_count = sum(1 for w in words if w in ENGLISH_MARKERS)

    so_ratio = so_count / len(words)
    en_ratio = en_count / len(words)

    if so_ratio > 0.08:
        lang = 'somali'
    elif en_ratio > 0.08:
        lang = 'english'
    else:
        lang = 'mixed'

    return (lang, round(so_ratio, 3), round(en_ratio, 3))


# ── Run on all raw files ──────────────────────────────────────────────────────
print(f'{"FILE":<35} {"TOTAL":>7} {"ENGLISH":>8} {"MIXED":>7} {"SOMALI":>8}')
print('─' * 70)

all_english = []

for fp in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(fp, dtype=str)
    df.columns = [c.lower() for c in df.columns]
    body_col = 'body' if 'body' in df.columns else 'content'

    df['_lang'], df['_so'], df['_en'] = zip(
        *df[body_col].apply(detect_language)
    )

    english_rows = df[df['_lang'] == 'english'].copy()
    english_rows['source_file'] = fp.name
    all_english.append(english_rows)

    n_en    = (df['_lang'] == 'english').sum()
    n_mixed = (df['_lang'] == 'mixed').sum()
    n_so    = (df['_lang'] == 'somali').sum()
    flag    = '  ⚠' if n_en > 0 else ''

    print(f'{fp.name:<35} {len(df):>7} {n_en:>8} {n_mixed:>7} {n_so:>8}{flag}')

# ── Show English article samples ──────────────────────────────────────────────
combined_english = pd.concat([df for df in all_english if len(df) > 0], ignore_index=True)

print(f'\nTotal English articles found: {len(combined_english):,}')
print('\nSample English articles:')
print('─' * 70)
for _, row in combined_english.sample(min(5, len(combined_english)), random_state=1).iterrows():
    title_col = 'headline' if 'headline' in row.index else 'title'
    body_col  = 'body' if 'body' in row.index else 'content'
    print(f'  FILE     : {row["source_file"]}')
    print(f'  TITLE    : {str(row[title_col])[:80]}')
    print(f'  BODY     : {str(row[body_col])[:150]}')
    print(f'  so_ratio={row["_so"]}  en_ratio={row["_en"]}')
    print()

# Save for review
combined_english.to_csv('data/review/english_articles_found.csv', index=False)
print('Saved → data/review/english_articles_found.csv')

FILE                                  TOTAL  ENGLISH   MIXED   SOMALI
──────────────────────────────────────────────────────────────────────
bbc_somali__ciyaaro.csv                 245        4       1      240  ⚠
caasimada__caalamka.csv                4049        3       0     4046  ⚠
caasimada__siyaasad.csv                5316        5       0     5311  ⚠
garoweonline__ciyaaro.csv                11       11       0        0  ⚠
goobjoog__amni.csv                     6595        1       1     6593  ⚠
goobjoog__caalamka.csv                 1519        1       0     1518  ⚠
goobjoog__siyaasad.csv                 4746        5       4     4737  ⚠
kooxda__ciyaaro.csv                    7193        4      24     7165  ⚠
kooxdamanta__ciyaaro.csv                  8        0       0        8
kubadlive__ciyaaro.csv                  114        1       1      112  ⚠
laacibnet__ciyaaro.csv                 1094       46       0     1048  ⚠
mustaqbalmedia__caalamka.csv           4548        1       

In [5]:
!pip install langdetect langid

## Cell 3 — Raw Data Inspection Report

In [6]:
"""
CELL 3 — RAW DATA INSPECTION
==============================
Prints a full report on the raw data before any cleaning.
Read this output carefully — it tells you:
  - How many rows exist per category and per site
  - How many headlines and bodies are missing
  - The length distribution of headlines and bodies
  - How many exact duplicate URLs and headlines exist

Nothing is modified in this cell — it is read-only inspection.
"""

df = raw_df.copy()

# Compute length metrics on the raw data
df['_hlen']   = df['headline'].astype(str).str.len()
df['_bwords'] = df['body'].astype(str).str.split().str.len()

print('═' * 65)
print('  RAW DATA INSPECTION REPORT')
print('═' * 65)
print(f'  Total rows loaded : {len(df):,}')

# ── Missing values ────────────────────────────────────────────────────────────
print('\n  ── Missing values ──────────────────────────────────────────')
for col in ['headline', 'body', 'url', 'category', 'site']:
    n = df[col].isna().sum() + (df[col].astype(str).str.strip() == '').sum()
    flag = '⚠' if n > 0 else ' '
    print(f'  {flag} {col:<15}: {n:>6,} missing ({n/len(df)*100:.1f}%)')

# ── Category distribution ─────────────────────────────────────────────────────
print('\n  ── Articles per category ───────────────────────────────────')
for cat, n in df['category'].value_counts().items():
    bar = '█' * int(n / df['category'].value_counts().max() * 25)
    print(f'  {cat:<15} {n:>6,}  {bar}')

# ── Site distribution ─────────────────────────────────────────────────────────
print('\n  ── Articles per site ───────────────────────────────────────')
for site, n in df['site'].value_counts().items():
    print(f'  {site:<22} {n:>6,}')

# ── Headline length ───────────────────────────────────────────────────────────
print('\n  ── Headline length (chars) ─────────────────────────────────')
print(df['_hlen'].describe().round(1).to_string())
print(f'  Too short (< {MIN_HEADLINE_CHARS})  : {(df["_hlen"] < MIN_HEADLINE_CHARS).sum():,}')
print(f'  Too long  (> {MAX_HEADLINE_CHARS}) : {(df["_hlen"] > MAX_HEADLINE_CHARS).sum():,}')

# ── Body word count ───────────────────────────────────────────────────────────
print('\n  ── Body word count ─────────────────────────────────────────')
print(df['_bwords'].describe().round(1).to_string())
print(f'  Too short (< {MIN_BODY_WORDS} words): {(df["_bwords"] < MIN_BODY_WORDS).sum():,}')

# ── Duplicates ────────────────────────────────────────────────────────────────
print('\n  ── Duplicates ──────────────────────────────────────────────')
print(f'  Duplicate URLs      : {df.duplicated(subset=["url"]).sum():,}')
print(f'  Duplicate headlines : {df.duplicated(subset=["headline"]).sum():,}')
print('\n' + '═' * 65)

═════════════════════════════════════════════════════════════════
  RAW DATA INSPECTION REPORT
═════════════════════════════════════════════════════════════════
  Total rows loaded : 35,797

  ── Missing values ──────────────────────────────────────────
    headline       :      0 missing (0.0%)
    body           :      0 missing (0.0%)
    url            :      0 missing (0.0%)
    category       :      0 missing (0.0%)
    site           :      0 missing (0.0%)

  ── Articles per category ───────────────────────────────────
  caalamka        10,116  █████████████████████████
  siyaasad        10,062  ████████████████████████
  ciyaaro          9,024  ██████████████████████
  amni             6,595  ████████████████

  ── Articles per site ───────────────────────────────────────
  goobjoog               12,860
  caasimada               9,365
  kooxda                  7,193
  mustaqbalmedia          4,640
  laacibnet               1,094
  bbc_somali                245
  puntlandpost  

## Cell 4 — All Cleaning Functions

In [9]:
"""
CELL 4 — ALL CLEANING FUNCTIONS
================================
This is the core of the cleaning pipeline.
Run this cell ONCE — every cell after this calls these functions.

Functions are grouped by what they do:
  A. Imports and safe defaults
  B. Text normalisation
  C. Article-level filters
  D. Headline cleaning
  E. Body cleaning
  F. Site dispatch table
  G. Suspicious checks
  H. Helpers and language detection
"""

# ══════════════════════════════════════════════════════════════════════════════
# A. IMPORTS AND SAFE DEFAULTS
# ══════════════════════════════════════════════════════════════════════════════

import re
import hashlib
from collections import Counter

# Optional libraries for stronger Somali language detection
try:
    from langdetect import detect, LangDetectException
except Exception:
    detect = None

    class LangDetectException(Exception):
        pass

try:
    import langid
except Exception:
    langid = None


# Safe fallback values if these were not defined in earlier config cells
MIN_HEADLINE_CHARS = globals().get("MIN_HEADLINE_CHARS", 8)
MAX_HEADLINE_CHARS = globals().get("MAX_HEADLINE_CHARS", 180)
MIN_BODY_CHARS = globals().get("MIN_BODY_CHARS", 80)
MIN_BODY_WORDS = globals().get("MIN_BODY_WORDS", 20)

VALID_CATEGORIES = globals().get(
    "VALID_CATEGORIES",
    {"siyaasad", "ciyaaro", "caafimaad", "caalamka", "amni"}
)


# ══════════════════════════════════════════════════════════════════════════════
# B. TEXT NORMALISATION
# ══════════════════════════════════════════════════════════════════════════════

def normalize_whitespace(text: str) -> str:
    """
    Collapses all forms of whitespace into a single space.
    Removes zero-width characters that are invisible but cause text issues.
    Found in: all sites.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(r"[\r\n\t\xa0]+", " ", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff\u00ad]", "", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def fix_punctuation_spacing(text: str) -> str:
    """
    Fixes spacing around punctuation marks.
    Example: 'dalka , Muqdisho' → 'dalka, Muqdisho'
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(r" +([.,;:!?])", r"\1", text)
    text = re.sub(r"([.,;:!?]) {2,}", r"\1 ", text)
    return text


# ══════════════════════════════════════════════════════════════════════════════
# C. ARTICLE-LEVEL FILTERS
# ══════════════════════════════════════════════════════════════════════════════

GOOBJOOG_NAV_TITLES = {
    "Wararka Caalamka", "Dhaqanka & Buugta", "Aragtida Xorta Ah",
    "Arrimaha Ganacsiga", "La-dagaallanka Al-Shabaab", "Deegaanka & Cimilada",
    "Arrimaha Amniga", "Dagaalka Dib-u-Xoreynta", "Siyaasadda Xogdhawrka",
    "Wararka Maanta", "Ciyaaraha", "Caafimaad", "Diinta",
}


def is_goobjoog_nav(headline: str) -> bool:
    """
    Returns True if this row is a Goobjoog sidebar navigation item,
    not a real news article.
    """
    return str(headline).strip() in GOOBJOOG_NAV_TITLES


def is_bbc_nav(headline: str, body: str) -> bool:
    """
    Returns True if this is a BBC site-level page, not a real article.
    """
    nav_patterns = [
        r"^BBC News\s*,",
        r"Learn how the BBC is working",
        r"BBC World Service",
    ]

    hl = str(headline)

    for pat in nav_patterns:
        if re.search(pat, hl, re.IGNORECASE):
            return True

    return False


def is_garoweonline_nav(body: str) -> bool:
    """
    Returns True if this is a GaroweOnline listing/navigation page.
    """
    return bool(
        re.search(
            r"Advertise with GaroweOnline|manage your cookie choices",
            str(body),
            re.IGNORECASE
        )
    )


def is_wararka24_tos(headline: str) -> bool:
    """
    Returns True if this row is a Wararka24 legal/TOS page.
    """
    tos_titles = {"Terms and Conditions", "Editorial Policy", "Privacy Policy"}
    return str(headline).strip() in tos_titles


def is_mustaqbal_contact(body: str) -> bool:
    """
    Returns True if this row is a Mustaqbal contact form page.
    """
    return bool(
        re.search(
            r"Your name.{0,30}Your email|nala soo xiriir.{0,50}email",
            str(body),
            re.IGNORECASE | re.DOTALL
        )
    )


def is_mustaqbal_privacy(body: str) -> bool:
    """
    Returns True if this row is a Mustaqbal privacy/CCPA policy page.
    """
    return bool(
        re.search(
            r"Xuquuqda Gaarka ah ee CCPA|Ha Iibiina Macluumaadkayga",
            str(body),
            re.IGNORECASE
        )
    )


def is_kooxdamanta_archive(headline: str) -> bool:
    """
    Returns True if this is a Kooxdamanta date archive page.
    """
    return bool(
        re.match(
            r"^Month:\s+\w+\s+\d{4}$",
            str(headline).strip()
        )
    )


def should_remove_entire_article(headline: str, body: str):
    """
    Master article-level filter.
    Returns a reason string if article should be removed, otherwise None.
    """
    if is_goobjoog_nav(headline):
        return "goobjoog_nav_item"

    if is_bbc_nav(headline, body):
        return "bbc_nav_page"

    if is_garoweonline_nav(body):
        return "garoweonline_listing_page"

    if is_wararka24_tos(headline):
        return "wararka24_tos_page"

    if is_mustaqbal_contact(body):
        return "mustaqbal_contact_page"

    if is_mustaqbal_privacy(body):
        return "mustaqbal_privacy_page"

    if is_kooxdamanta_archive(headline):
        return "kooxdamanta_archive_page"

    return None


# ══════════════════════════════════════════════════════════════════════════════
# D. HEADLINE CLEANING
# ══════════════════════════════════════════════════════════════════════════════

def clean_headline(text: str) -> str:
    """
    Cleans a single headline string.
    """
    if not isinstance(text, str):
        return ""

    text = normalize_whitespace(text)

    # Remove media labels at the start of headlines
    text = re.sub(
        r"^(Sawirro|Sawir|Daawo|DAAWO|Muqaal|VIDEO|PHOTO|Akhriso|AKHRISO|Akhri|Maqal)\s*[:\-]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove site name appended at the end
    text = re.sub(
        r"\s*[|\-–—]\s*(Caasimada(\s+Online)?|Goobjoog(\s+News)?|"
        r"Mustaqbal(\s+Media)?|Radio\s+Dalsan|Hiiraan|BBC|SONNA).*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove wrapping quotation marks
    text = re.sub(
        r"^[\"\u201c\u201e'\u2018](.+)[\"\u201d'\u2019]$",
        r"\1",
        text
    )

    text = fix_punctuation_spacing(text)
    return normalize_whitespace(text)


# ══════════════════════════════════════════════════════════════════════════════
# E. BODY CLEANING FUNCTIONS
# IMPORTANT: all functions must be defined BEFORE SITE_CLEANERS
# ══════════════════════════════════════════════════════════════════════════════

def remove_bbc_image_captions(text: str) -> str:
    """
    Removes BBC Somali image credit tags from article bodies.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"Xigashada Sawirka\s*,?\s*"
        r"(?:"
            r"Getty Images?"
            r"|GETTY IMAGES?"
            r"|Reuters"
            r"|AFP(?: via \w+)?"
            r"|EPA"
            r"|AP"
            r"|PA(?: Media)?"
            r"|BBC(?: Via [\w\s]+?(?=\s[A-Z]))?"
            r"|FIFA"
            r"|SONNA"
            r"|SOCIAL(?: MEDIA)?"
            r"|Social(?: Media)?"
            r"|Facebook(?:/[\w]+)?"
            r"|Instagram"
            r"|X(?=\s)"
            r"|[A-Z]{2,}(?:\s+[A-Z]{2,})*"
            r"|[A-Z][a-z]+(?:\s[A-Z][a-z]+){0,3}"
            r"|G\.D\."
            r"|Cafonline"
            r"|Villa"
        r")?\s*\.?\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return re.sub(r" {2,}", " ", text).strip()


def remove_bbc_end_of_content(text: str) -> str:
    """
    Removes BBC 'End of content' and everything after it.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"End of content.*$",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    return text


def remove_caasimada_share_dateline(text: str) -> str:
    """
    Removes Caasimada dateline prefix.

    Pattern:
      'Share <City> (Caasimada Online) – <article text>'
    """
    if not isinstance(text, str):
        return ""

    return re.sub(
        r"^Share\s+.{0,100}?[–\-—]+\s*",
        "",
        text.strip()
    )


def remove_caasimada_online_byline(text: str) -> str:
    """
    Removes Caasimada Online site bylines and CTAs from the end of articles.
    """
    if not isinstance(text, str):
        return ""

    # Remove video/watch link before byline
    text = re.sub(
        r"\s*Halkan\s+[Kk]a\s+[Dd]aawo\s+warark\w*(?:\s+maanta)?\s+oo\s+muuqaal\s+ah\s+",
        " ",
        text
    )

    # Remove main Caasimada Online footer/byline block
    text = re.sub(
        r"\s*(?:HALKAN\s+CLICK\s+SII\s+)?"
        r"Caasimada\s+Online"
        r"(?:\s+Xafiiska\s+[\w\s]{1,40})?"
        r"(?:[\s.]*HALKAN\s+CLICK\s+SII)?"
        r"(?:,\s*waa\s+mareeg[^.]*\.?\s*"
        r"Kusoo\s+dir[^.]*\.?\s*Mahadsanid\.?)?"
        r"(?:\s+waxay\s+leedahay\s+App[^.]*\.)?"
        r"(?:\s*Mahadsanid\.?)?\s*$",
        "",
        text.strip(),
        flags=re.IGNORECASE
    )

    return text.strip()


def remove_caasimada_wq_credit(text: str) -> str:
    """
    Removes W/Q or W/T credits from Caasimada articles.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"\s*W/[QT]\s*:\s*[^\n.]{0,160}",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text.strip()


def remove_trailing_source_credits(text: str) -> str:
    """
    Removes trailing wire/source credits.

    Examples:
      AP, VOA
      AFP + BBC Somali
      Reuters
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()

    # Wire service credits
    text = re.sub(
        r"\s*(?:AP|VOA|AFP|Reuters|BBC)(?:\s*[,+/&]\s*"
        r"(?:AP|VOA|AFP|Reuters|BBC|Somali|SOMALI))*\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Journalist contact invitation
    text = re.sub(
        r"\s*[\w\s]{4,40}[–\-]\s*Kala xiriir:\s*$",
        "",
        text.strip(),
        flags=re.IGNORECASE
    )

    return text.strip()


def remove_caasimada_read_more(text: str) -> str:
    """
    Removes Caasimada 'Read more', 'Isha:', emails, and video prompts.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(r"\s*Read more\s*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Isha\s*:\s*\S+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\[email[^\]]*\]|\S+@\S+\.\S+", " ", text)
    text = re.sub(
        r"Hoos\s+ka\s+daawo[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_goobjoog_footer(text: str) -> str:
    """
    Removes Goobjoog video/audio prompts and footer byline.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"(Halkaan\s+hoose\s+ka\s+daawo|Hoos\s+Ka\s+Dhageyso|"
        r"HALKA\s+HOOSE\s+KA\s+DHAGEYSO)[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*Goobjoog\s+News\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Wariye\s*:\s*[^.]{0,60}\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_mustaqbal_footer(text: str) -> str:
    """
    Removes Mustaqbal footer/follow prompts.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"La\s+soco\s+wixii\s+kasoo\s+kordha[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*Mustaqbal\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_mustaqbal_isha(text: str) -> str:
    """
    Removes Mustaqbal source attribution at the end.

    Examples:
      Isha: VOA
      Isha: BBC SOMALI
      Isha: AFP / VOA Somali
    """
    if not isinstance(text, str):
        return ""

    return re.sub(
        r"\s*Isha\s*:\s*[A-Za-z][A-Za-z\s/&+\.]{1,80}$",
        "",
        text.strip(),
        flags=re.IGNORECASE
    )


def remove_kooxda_footer(text: str) -> str:
    """
    Removes Kooxda.com footer/byline/photo attribution.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"Kooxda\.com\s+ayaa\s+xusaysa[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"La\s+soco\s+Kooxda\.com[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Sawirro?\s+la\s+qaaday[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_social_media_lines(text: str) -> str:
    """
    Removes social media call-to-action sentences.
    """
    if not isinstance(text, str):
        return ""

    social_cta = re.compile(
        r"(raac|subscribe|nagula soo xiriir|nala soco|follow us|"
        r"la soco barnaamijka|telegram|whatsapp channel)",
        re.IGNORECASE
    )

    sentences = re.split(r"(?<=[.!?]) +", text)
    clean = [s for s in sentences if not social_cta.search(s)]

    return " ".join(clean)


def remove_kubadlive_view_counter(text: str) -> str:
    """
    Removes Kubadlive view counters.
    Example:
      Views: 9
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"\s*Views?\s*:\s*\d+\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_puntlandpost_byline(text: str) -> str:
    """
    Removes trailing Puntland Post byline.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(
        r"\s*PUNTLAND\s+POST\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_ellipsis_tails(text: str) -> str:
    """
    Removes trailing ellipsis from teaser/listing content.
    """
    if not isinstance(text, str):
        return ""

    text = re.sub(r"\s*[…]+\s*$", "", text)
    text = re.sub(r"\s*\.{3,}\s*$", "", text)

    return text.strip()


# ══════════════════════════════════════════════════════════════════════════════
# F. SITE DISPATCH TABLE
# Now safe because all cleaner functions above are already defined.
# ══════════════════════════════════════════════════════════════════════════════

SITE_CLEANERS = {
    "bbc_somali": [
        remove_bbc_image_captions,
        remove_bbc_end_of_content,
    ],

    "caasimada": [
        remove_caasimada_share_dateline,
        remove_caasimada_online_byline,
        remove_caasimada_wq_credit,
        remove_trailing_source_credits,
        remove_caasimada_read_more,
    ],

    "mustaqbalmedia": [
        remove_mustaqbal_footer,
        remove_mustaqbal_isha,
    ],

    "goobjoog": [
        remove_goobjoog_footer,
    ],

    "kooxda": [
        remove_kooxda_footer,
    ],

    "kubadlive": [
        remove_kubadlive_view_counter,
    ],

    "puntlandpost": [
        remove_puntlandpost_byline,
    ],
}


def clean_body(text: str, site: str = "") -> str:
    """
    Full body cleaning pipeline for one article.

    Step 1: Run site-specific cleaners
    Step 2: Remove social media CTA lines
    Step 3: Remove trailing ellipsis
    Step 4: Fix punctuation spacing
    Step 5: Final whitespace normalisation
    """
    if not isinstance(text, str):
        return ""

    site = str(site).strip().lower()

    if site in SITE_CLEANERS:
        for fn in SITE_CLEANERS[site]:
            text = fn(text)

    text = remove_social_media_lines(text)
    text = remove_ellipsis_tails(text)
    text = fix_punctuation_spacing(text)
    text = normalize_whitespace(text)

    return text


# ══════════════════════════════════════════════════════════════════════════════
# G. SUSPICIOUS ARTICLE CHECKS
# ══════════════════════════════════════════════════════════════════════════════

def get_suspicious_flags(row) -> list:
    """
    Checks a cleaned row for quality issues.
    Returns a list of flag strings.
    Empty list means the row is clean.
    """
    flags = []

    hl = str(row.get("headline_clean", ""))
    bd = str(row.get("body_clean", ""))
    bwords = len(bd.split())

    if len(hl) < MIN_HEADLINE_CHARS:
        flags.append(f"short_headline({len(hl)}chars)")

    if len(hl) > MAX_HEADLINE_CHARS:
        flags.append(f"long_headline({len(hl)}chars)")

    if len(bd) < MIN_BODY_CHARS:
        flags.append(f"short_body({len(bd)}chars)")

    if bwords < MIN_BODY_WORDS:
        flags.append(f"few_body_words({bwords})")

    if not row.get("url") or str(row.get("url", "")) in ("nan", "unknown", ""):
        flags.append("missing_url")

    if str(row.get("category", "")) not in VALID_CATEGORIES:
        flags.append(f'unexpected_category({row.get("category", "")})')

    # Repetitive content check
    if bwords > 15:
        top_word_count = Counter(bd.lower().split()).most_common(1)[0][1]

        if top_word_count / bwords > 0.35:
            flags.append("repetitive_content")

    # English-heavy content check
    english_words = {
        "the", "and", "is", "in", "of", "to", "a", "that",
        "it", "for", "with", "on"
    }

    words = bd.lower().split()

    if words:
        en_ratio = sum(1 for w in words if w in english_words) / len(words)

        if en_ratio > 0.15:
            flags.append(f"possibly_english({en_ratio:.0%})")

    return flags


# ══════════════════════════════════════════════════════════════════════════════
# H. HELPERS AND LANGUAGE DETECTION
# ══════════════════════════════════════════════════════════════════════════════

def md5_hash(text: str) -> str:
    """
    Returns MD5 hash of a lowercased, stripped string.
    Used for deduplication.
    """
    return hashlib.md5(
        str(text).strip().lower().encode()
    ).hexdigest()


def print_dedup_step(label: str, before: int, after: int):
    """
    Prints a formatted before/after count for a deduplication step.
    """
    removed = before - after
    print(f"  {label:<40} removed {removed:>5,}  (→ {after:,} remain)")


def is_somali_article(headline: str, body: str) -> tuple:
    """
    Returns:
      (is_somali: bool, reason: str)

    Uses:
      1. Somali word heuristic
      2. langdetect if installed
      3. langid if installed

    Article is kept if any check says Somali.
    Article is removed only if all checks say non-Somali.
    """
    text = f"{str(headline)} {str(body)}"
    words = text.lower().split()

    if not words:
        return False, "empty"

    # Check 1: Somali word heuristic
    SOMALI_MARKERS = {
        "ayaa", "ayuu", "ayey", "waxaa", "waxuu", "waxay", "waxan",
        "oo", "iyo", "ee", "ku", "ka", "la", "uu", "ay",
        "in", "waa", "ah", "ama", "haddii", "sidoo", "kale",
        "waxa", "lagu", "looga", "ugu", "kaga", "iska",
        "dalka", "magaalada", "dowladda", "shacabka", "caalamka",
        "madaxweynaha", "wasaaradda", "ciidanka", "kooxda",
    }

    so_count = sum(1 for w in words if w in SOMALI_MARKERS)
    so_ratio = so_count / len(words)

    if so_ratio >= 0.08:
        return True, f"heuristic_somali({so_ratio:.2f})"

    # Check 2: langdetect
    if detect is not None:
        try:
            lang = detect(text[:1000])

            if lang == "so":
                return True, "langdetect_somali"

            # Somali is sometimes misdetected as nearby languages
            if lang in ("sw", "tl", "id", "ms"):
                return True, f"langdetect_near_somali({lang})"

        except LangDetectException:
            pass
        except Exception:
            pass

    # Check 3: langid
    if langid is not None:
        try:
            lang_id, confidence = langid.classify(text[:1000])

            if lang_id == "so":
                return True, f"langid_somali({confidence:.2f})"

        except Exception:
            pass

    english_words = {
        "the", "and", "is", "in", "of", "to", "a", "that",
        "it", "for", "with", "on", "are", "was", "has",
        "have", "this", "an", "at"
    }

    en_count = sum(1 for w in words if w in english_words)
    en_ratio = en_count / len(words)

    return False, f"non_somali(so={so_ratio:.2f},en={en_ratio:.2f})"


print("✅ All cleaning functions loaded successfully.")
print(f"   Site-specific cleaners configured for: {list(SITE_CLEANERS.keys())}")

✅ All cleaning functions loaded successfully.
   Site-specific cleaners configured for: ['bbc_somali', 'caasimada', 'mustaqbalmedia', 'goobjoog', 'kooxda', 'kubadlive', 'puntlandpost']


## Cell 5 — Remove Nav / TOS / Boilerplate Articles

In [10]:
"""
CELL 5 — REMOVE ENTIRE BAD ARTICLES
=====================================
Some rows are not real news articles at all.
They were accidentally scraped because the scraper followed links
to navigation pages, contact forms, privacy policy pages, etc.

This cell calls should_remove_entire_article() on every row.
Removed rows are saved to data/review/removed_bad_articles.csv
so you can inspect them manually.

Found in analysis:
  - Goobjoog: sidebar nav items (La-dagaallanka Al-Shabaab, etc.)
  - BBC: home page, category page, trust page (3 rows)
  - GaroweOnline: ALL 11 rows were date-archive listing pages
  - Wararka24: 2 rows were Terms & Conditions and Editorial Policy
  - Mustaqbal: contact page + CCPA privacy pages (184 rows)
  - Kooxdamanta: 1 'Month: May 2026' WordPress archive page
"""

df = raw_df.copy()
n_before = len(df)

# Apply the master filter to every row
print('Scanning for bad articles...')
df['_removal_reason'] = df.apply(
    lambda r: should_remove_entire_article(r['headline'], r['body']),
    axis=1
)

# Split into bad (to remove) and good (to keep)
bad_df  = df[df['_removal_reason'].notna()].copy()
df      = df[df['_removal_reason'].isna()].copy()
df      = df.drop(columns=['_removal_reason'])

# Report
print(f'\n  Removed {len(bad_df):,} bad articles:')
for reason, count in bad_df['_removal_reason'].value_counts().items():
    print(f'    {reason:<35}: {count:,}')

# Save for manual inspection
bad_df.to_csv(REVIEW_DIR / 'removed_bad_articles.csv', index=False)
print(f'\n  Saved → {REVIEW_DIR}/removed_bad_articles.csv')
print(f'  Rows remaining: {len(df):,}')

Scanning for bad articles...

  Removed 45 bad articles:
    goobjoog_nav_item                  : 19
    garoweonline_listing_page          : 11
    wararka24_tos_page                 : 6
    mustaqbal_contact_page             : 5
    bbc_nav_page                       : 3
    kooxdamanta_archive_page           : 1

  Saved → data\review/removed_bad_articles.csv
  Rows remaining: 35,752


In [11]:
"""
REMOVE NON-SOMALI ARTICLES
======================================
Removes articles that are not in Somali language.

Most common sources of English articles (from analysis):
  - GaroweOnline: all 11 ciyaaro articles were English
  - Wararka24: Terms & Conditions page in English
  - Puntlandpost: occasional English copyright/subscribe notices
  - BBC Somali: 3 nav pages were in English

Strategy: keep if heuristic OR library says Somali.
Remove only if ALL checks confirm non-Somali.
Non-Somali rows go to data/review/removed_non_somali.csv
"""

from langdetect import detect, LangDetectException
import langid

n_before = len(df)
print('Detecting article language...')

# Apply language check to every row
df['_is_somali'], df['_lang_reason'] = zip(
    *df.apply(
        lambda r: is_somali_article(r['headline'], r['body']),
        axis=1
    )
)

# Split
non_somali_df = df[~df['_is_somali']].copy()
df            = df[df['_is_somali']].copy()

# Report
print(f'\n  Non-Somali articles removed: {len(non_somali_df):,}')
if len(non_somali_df) > 0:
    print('\n  Breakdown by site:')
    print(non_somali_df['site'].value_counts().to_string())

    print('\n  Breakdown by reason:')
    reasons = non_somali_df['_lang_reason'].str.extract(r'^([a-z_]+)')[0]
    print(reasons.value_counts().to_string())

    print('\n  Samples:')
    for _, row in non_somali_df.head().iterrows():
        print(f'  SITE  : {row["site"]}')
        print(f'  TITLE : {str(row["headline"])[:80]}')
        print(f'  BODY  : {str(row["body"])[:120]}')
        print(f'  REASON: {row["_lang_reason"]}')
        print()

# Save for review
non_somali_df.to_csv(REVIEW_DIR / 'removed_non_somali.csv', index=False)
print(f'  Saved → {REVIEW_DIR}/removed_non_somali.csv')
print(f'  Remaining rows: {len(df):,}')

# Drop helper columns
df = df.drop(columns=['_is_somali', '_lang_reason'])

Detecting article language...

  Non-Somali articles removed: 93

  Breakdown by site:
site
laacibnet         45
kooxda            24
goobjoog           7
caasimada          6
puntlandpost       4
mustaqbalmedia     4
bbc_somali         2
kubadlive          1

  Breakdown by reason:
0
non_somali    93

  Samples:
  SITE  : bbc_somali
  TITLE : Take control of your cookies
  BODY  : Have a browse to see what we use them for and how you can change your settings to suit you.
  REASON: non_somali(so=0.00,en=0.28)

  SITE  : bbc_somali
  TITLE : Guidance: Links and feeds
  BODY  : The BBC's global reputation is based on its editorial integrity and independence. Our audiences need to be confident tha
  REASON: non_somali(so=0.01,en=0.27)

  SITE  : caasimada
  TITLE : Contact Us
  BODY  : Waxaad nagala soo xiriirtaa [email protected] Mahadsanid Caasimada Online is a reputable and reliable source of news and
  REASON: non_somali(so=0.04,en=0.24)

  SITE  : caasimada
  TITLE : Bookmark page
  

## Cell 6 — Clean Headlines

In [12]:
"""
CLEAN HEADLINES
==========================
Applies clean_headline() to every row.
Creates a new column 'headline_clean' — the original 'headline' is kept
so you can always compare before and after.

After cleaning, prints:
  - How many headlines were changed
  - Sample before/after comparisons
  - How many headlines are now empty (will be handled in Cell 8)
"""

print('Cleaning headlines...')
df['headline_clean'] = df['headline'].progress_apply(clean_headline)

# Find rows where something actually changed
changed = df[df['headline'] != df['headline_clean']]
print(f'\n  Headlines modified : {len(changed):,} out of {len(df):,}')

# Show before/after examples
if len(changed) > 0:
    print('\n  Sample changes:')
    print('  ' + '─' * 60)
    for _, row in changed.sample(min(10, len(changed)), random_state=42).iterrows():
        print(f'  BEFORE : {str(row["headline"])[:90]}')
        print(f'  AFTER  : {str(row["headline_clean"])[:90]}')
        print()

# Check for newly empty headlines
empty_after = (df['headline_clean'].str.strip() == '') | df['headline_clean'].isna()
print(f'  Empty after cleaning : {empty_after.sum()} (will be removed in Cell 8)')

Cleaning headlines...


Processing:   0%|          | 0/35659 [00:00<?, ?it/s]


  Headlines modified : 1,391 out of 35,659

  Sample changes:
  ────────────────────────────────────────────────────────────
  BEFORE : Baarlamaanka Yurub : Sacuudiga Iyo Imaaraadka Waxaa Ay Soomaaliya Ku Ciqaabeen Mowqifkeeda
  AFTER  : Baarlamaanka Yurub: Sacuudiga Iyo Imaaraadka Waxaa Ay Soomaaliya Ku Ciqaabeen Mowqifkeeda 

  BEFORE : Daawo: Xasan Sheekh oo shaaciyay in loo jiheysanayo badda
  AFTER  : Xasan Sheekh oo shaaciyay in loo jiheysanayo badda

  BEFORE : “Waa Dammiin, Isagaa La Degay Kooxda” :- Carragher Oo Si Aan Naxariis Lahayn Furka Ugu Tuu
  AFTER  : “Waa Dammiin, Isagaa La Degay Kooxda”:- Carragher Oo Si Aan Naxariis Lahayn Furka Ugu Tuur

  BEFORE : Sawirro: Qarax Xooggan Oo Lagu Weeraray Xarunta Degmada Howlwadaag
  AFTER  : Qarax Xooggan Oo Lagu Weeraray Xarunta Degmada Howlwadaag

  BEFORE : Daawo: Xasan Macallin oo diiday inuu raali-gelin ka bixiyo arrintii carada weyn dhalisay
  AFTER  : Xasan Macallin oo diiday inuu raali-gelin ka bixiyo arrintii carada weyn 

## Cell 7 — Clean Article Bodies

In [13]:
"""
CLEAN ARTICLE BODIES
===============================
Applies clean_body() to every row, passing the site name so
site-specific cleaners only run on the right articles.

Creates 'body_clean' column — the original 'body' is kept.

This cell may take 1-2 minutes depending on dataset size.
A progress bar is shown.

After cleaning, prints:
  - Word count statistics before vs after
  - Bytes removed per site (shows which cleaners worked hardest)
  - A sample cleaned article
"""

print('Cleaning article bodies (this may take 1-2 minutes)...')

# Pass both the body text and the site name so site dispatch works
df['body_clean'] = df.progress_apply(
    lambda r: clean_body(str(r['body']), site=str(r['site'])),
    axis=1
)

# Compute word counts before and after
df['_bwords_raw']   = df['body'].astype(str).str.split().str.len()
df['_bwords_clean'] = df['body_clean'].astype(str).str.split().str.len()
df['_bytes_removed'] = (df['body'].astype(str).str.len() -
                        df['body_clean'].astype(str).str.len())

print('\n  ── Word count: raw vs clean ─────────────────────────────────')
# print(f'  {'Metric':<12} {'Raw':>10} {'Cleaned':>10}')
# print(f'  {'='*34}')
for stat in ['mean', 'min', 'max']:
    raw_v   = getattr(df['_bwords_raw'],   stat)()
    clean_v = getattr(df['_bwords_clean'], stat)()
    print(f'  {stat:<12} {raw_v:>10.0f} {clean_v:>10.0f}')

print('\n  ── Bytes removed per site ───────────────────────────────────')
site_bytes = df.groupby('site')['_bytes_removed'].mean().sort_values(ascending=False)
for site, avg_bytes in site_bytes.items():
    print(f'  {site:<22}: avg {avg_bytes:>6.0f} bytes removed per article')

print('\n  ── Sample cleaned article ───────────────────────────────────')
sample = df[df['_bwords_clean'] > 80].sample(1, random_state=7).iloc[0]
print(f'  SITE     : {sample["site"]}')
print(f'  CATEGORY : {sample["category"]}')
print(f'  HEADLINE : {sample["headline_clean"]}')
print(f'  BODY     : {sample["body_clean"][:400]}...')

Cleaning article bodies (this may take 1-2 minutes)...


Processing:   0%|          | 0/35659 [00:00<?, ?it/s]


  ── Word count: raw vs clean ─────────────────────────────────
  mean                240        227
  min                   9          0
  max                2692       2684

  ── Bytes removed per site ───────────────────────────────────
  bbc_somali            : avg   2128 bytes removed per article
  caasimada             : avg    123 bytes removed per article
  puntlandpost          : avg     93 bytes removed per article
  mustaqbalmedia        : avg     51 bytes removed per article
  goobjoog              : avg     44 bytes removed per article
  kooxda                : avg     41 bytes removed per article
  kubadlive             : avg     38 bytes removed per article
  laacibnet             : avg     35 bytes removed per article
  wararka24             : avg     25 bytes removed per article
  sonna                 : avg     21 bytes removed per article
  kooxdamanta           : avg      2 bytes removed per article

  ── Sample cleaned article ───────────────────────────────────
 

## Cell 8 — Handle Missing Values

In [14]:
"""
HANDLE MISSING VALUES
================================
Standardises all empty/null representations and removes rows
that cannot be used for NLP training (missing headline or body).

Strategy:
  - Missing headline or body → remove (useless for training)
  - Missing URL → replace with 'unknown' (not critical for training)
  - Missing category → flag and investigate

Removed rows are saved to data/review/removed_missing.csv
"""

n_before = len(df)

# Standardise all flavours of empty strings
for col in ['headline_clean', 'body_clean', 'url', 'category', 'site']:
    if col in df.columns:
        df[col] = df[col].replace(['nan', 'None', 'NaN', ''], np.nan)

# Report missing values before removal
print('  Missing value counts (after cleaning):')
for col in ['headline_clean', 'body_clean', 'url', 'category']:
    n = df[col].isna().sum()
    flag = '⚠' if n > 0 else ' '
    print(f'  {flag} {col:<20}: {n:>5,}')

# Remove rows with no usable headline or body
critical_mask = df['headline_clean'].isna() | df['body_clean'].isna()
missing_rows  = df[critical_mask].copy()
missing_rows['removal_reason'] = 'missing_headline_or_body_after_cleaning'
missing_rows.to_csv(REVIEW_DIR / 'removed_missing.csv', index=False)
df = df[~critical_mask].copy()

# Replace missing URLs with placeholder
df['url'] = df['url'].fillna('unknown')

print(f'\n  Removed (no headline/body) : {len(missing_rows):,}')
print(f'  Saved  → {REVIEW_DIR}/removed_missing.csv')
print(f'  Remaining rows             : {len(df):,}')

  Missing value counts (after cleaning):
    headline_clean      :     0
  ⚠ body_clean          :     8
    url                 :     0
    category            :     0

  Removed (no headline/body) : 8
  Saved  → data\review/removed_missing.csv
  Remaining rows             : 35,651


## Cell 9 — Deduplicate

In [15]:
"""
DEDUPLICATION
========================
Removes duplicate articles in three stages:

Stage 1 — Exact URL duplicates
  The same URL was scraped from two different files (e.g. the
  same article appearing in siyaasad and caalamka listings).
  Found: goobjoog__amni (48 dups), caasimada__siyaasad (5 dups).

Stage 2 — Exact headline duplicates
  Same headline text, possibly different URL (article mirrored).
  Uses MD5 hash of lowercased headline for fast comparison.

Stage 3 — Near-duplicate bodies
  Hash the first 500 characters of the cleaned body.
  This catches articles that were re-published with a slightly
  different headline or URL but identical content.
  500 chars is enough to fingerprint uniquely without being
  sensitive to minor text differences.

All removed duplicates are saved to data/review/removed_duplicates.csv
"""

n_start   = len(df)
dup_rows  = []

# ── Stage 1: Exact URL ────────────────────────────────────────────────────────
n1 = len(df)
url_dup_mask = df.duplicated(subset=['url'], keep='first') & (df['url'] != 'unknown')
dup_rows.append(df[url_dup_mask].assign(dup_stage='1_exact_url'))
df = df[~url_dup_mask].copy()
print_dedup_step('Stage 1: Exact duplicate URLs', n1, len(df))

# ── Stage 2: Exact headline hash ──────────────────────────────────────────────
n2 = len(df)
df['_hl_hash'] = df['headline_clean'].apply(md5_hash)
hl_dup_mask    = df.duplicated(subset=['_hl_hash'], keep='first')
dup_rows.append(df[hl_dup_mask].assign(dup_stage='2_exact_headline'))
df = df[~hl_dup_mask].copy()
print_dedup_step('Stage 2: Exact duplicate headlines', n2, len(df))

# ── Stage 3: Near-duplicate body (first 500 chars) ────────────────────────────
n3 = len(df)
df['_body_hash'] = df['body_clean'].apply(lambda x: md5_hash(str(x)[:500]))
body_dup_mask    = df.duplicated(subset=['_body_hash'], keep='first')
dup_rows.append(df[body_dup_mask].assign(dup_stage='3_near_duplicate_body'))
df = df[~body_dup_mask].copy()
print_dedup_step('Stage 3: Near-duplicate bodies (500-char hash)', n3, len(df))

# Save all removed duplicates for reference
if dup_rows:
    pd.concat(dup_rows, ignore_index=True).to_csv(
        REVIEW_DIR / 'removed_duplicates.csv', index=False
    )

total_removed = n_start - len(df)
print(f'\n  Total duplicates removed : {total_removed:,}')
print(f'  Remaining after dedup    : {len(df):,}')
print(f'  Saved → {REVIEW_DIR}/removed_duplicates.csv')

  Stage 1: Exact duplicate URLs            removed   691  (→ 34,960 remain)
  Stage 2: Exact duplicate headlines       removed    39  (→ 34,921 remain)
  Stage 3: Near-duplicate bodies (500-char hash) removed   139  (→ 34,782 remain)

  Total duplicates removed : 869
  Remaining after dedup    : 34,782
  Saved → data\review/removed_duplicates.csv


## Cell 10 — Suspicious Article Detection

In [16]:
"""
SUSPICIOUS ARTICLE DETECTION
========================================
Flags articles that pass the earlier filters but still have quality issues.
These are NOT automatically deleted — they go to a review file.

Flags checked (defined in get_suspicious_flags() in Cell 4):
  - short_headline      : headline < 15 chars
  - long_headline       : headline > 300 chars (probably a scraped paragraph)
  - short_body          : body < 150 chars
  - few_body_words      : body < 30 words
  - missing_url         : no URL recorded
  - unexpected_category : category not in VALID_CATEGORIES
  - repetitive_content  : one word > 35% of all words (boilerplate list)
  - possibly_english    : > 15% common English function words

The 'possibly_english' flag is important because GaroweOnline's
articles were English (the site publishes in both languages).
"""

print('Scanning for suspicious articles...')

df['_flags']    = df.apply(get_suspicious_flags, axis=1)
df['_flag_str'] = df['_flags'].apply(lambda f: ' | '.join(f))

suspicious_mask = df['_flag_str'] != ''
suspicious_df   = df[suspicious_mask].copy()
clean_df        = df[~suspicious_mask].copy()

print(f'  Flagged for review : {len(suspicious_df):,}')
print(f'  Clean articles     : {len(clean_df):,}')

# Flag breakdown
print('\n  Flag breakdown:')
all_flags   = [f for flags in df['_flags'] for f in flags]
flag_prefix = re.compile(r'^([a-z_]+)')
flag_counts = Counter(
    flag_prefix.match(f).group(1) for f in all_flags if flag_prefix.match(f)
)
for flag, count in flag_counts.most_common():
    print(f'  {flag:<30}: {count:,}')

# Show samples
if len(suspicious_df) > 0:
    print('\n  Sample suspicious rows:')
    for _, row in suspicious_df.sample(min(4, len(suspicious_df)), random_state=1).iterrows():
        print(f'  FLAG     : {row["_flag_str"]}')
        print(f'  HEADLINE : {str(row.get("headline_clean", ""))[:80]}')
        print(f'  BODY     : {str(row.get("body_clean", ""))[:120]}')
        print()

# Save for manual review
review_cols = ['site', 'category', 'url', 'headline_clean', 'body_clean', '_flag_str', 'source_file']
suspicious_df[[c for c in review_cols if c in suspicious_df.columns]].to_csv(
    REVIEW_DIR / 'suspicious_articles.csv', index=False
)
print(f'  Saved → {REVIEW_DIR}/suspicious_articles.csv')
print('  Review this file manually before deciding to include or exclude these rows.')

Scanning for suspicious articles...
  Flagged for review : 133
  Clean articles     : 34,649

  Flag breakdown:
  few_body_words                : 127
  short_body                    : 57
  short_headline                : 6

  Sample suspicious rows:
  FLAG     : short_headline(8chars)
  HEADLINE : Muuqaal:
  BODY     : 5 Maalin ayaa laga joogaa markii Djibouti lagu soo gabagabeeyey wadahaddallada Soomaaliya iyo Somaliland, sidaas ay taha

  FLAG     : few_body_words(22)
  HEADLINE : Qaraxa Ka Dhacay Muqdisho
  BODY     : Halkan waa goobta qaraxa gaaro ee agagaarka Sayidka ee Muqdisho, waxaa jira khasaare maaliyadeed Weli lama hayo faahfaah

  FLAG     : few_body_words(22)
  HEADLINE : Abuukaate Daahir Carab“Eeddii laga keenay Weriye Cabdicasiis Gurbiye waxaan ku s
  BODY     : “ Eeddii laga keenay Weriye Cabdicasiis Gurbiye waxaan ku sameynay Is-hortaag sharciga waafaqsan ’’ Sidaas waxa saxaafad

  FLAG     : few_body_words(29)
  HEADLINE : Madaxweyne Erdogan oo ka digay qorshe ka dhan

## Cell 11 — Category & Source Validation

In [17]:
"""
CATEGORY & SOURCE VALIDATION
========================================
Verifies the category and site distributions in the clean dataset.

Checks:
  - Are all categories in VALID_CATEGORIES?
  - Is there a balance issue (one category much larger than others)?
  - Which sites contributed how much to each category?

Nothing is removed here — this is diagnostic output.
If category balance is very uneven, you may want to downsample
the larger categories before model training.
"""

print('═' * 65)
print('  CATEGORY & SOURCE VALIDATION')
print('═' * 65)

# ── Category distribution ─────────────────────────────────────────────────────
print('\n  Category distribution (clean dataset):')
total = len(clean_df)
cat_counts = clean_df['category'].value_counts()
for cat, n in cat_counts.items():
    bar = '█' * int(n / cat_counts.max() * 30)
    pct = n / total * 100
    ok  = '✔' if cat in VALID_CATEGORIES else '⚠'
    print(f'  {ok} {cat:<15} {n:>7,}  {pct:>5.1f}%  {bar}')

# Check for unexpected categories
unexpected = clean_df[~clean_df['category'].isin(VALID_CATEGORIES)]
if len(unexpected):
    print(f'\n  ⚠ Rows with unexpected categories: {len(unexpected)}')
    print(unexpected['category'].value_counts().to_string())

# ── Source distribution ───────────────────────────────────────────────────────
print('\n  Source distribution (clean dataset):')
src_counts = clean_df['site'].value_counts()
for site, n in src_counts.items():
    bar = '█' * int(n / src_counts.max() * 30)
    pct = n / total * 100
    print(f'  {site:<22} {n:>7,}  {pct:>5.1f}%  {bar}')

# ── Category × Site cross-table ───────────────────────────────────────────────
print('\n  Articles per category per site:')
pivot = clean_df.groupby(['category', 'site']).size().unstack(fill_value=0)
print(pivot.to_string())

# ── Class balance warning ─────────────────────────────────────────────────────
if len(cat_counts) > 1:
    ratio = cat_counts.max() / cat_counts.min()
    if ratio > 3:
        print(f'\n  ⚠ Class imbalance detected: largest category is {ratio:.1f}x bigger than smallest.')
        print('    Consider downsampling before model training.')
    else:
        print(f'\n  ✔ Class balance ratio: {ratio:.1f}x (acceptable)')

print('\n' + '═' * 65)

═════════════════════════════════════════════════════════════════
  CATEGORY & SOURCE VALIDATION
═════════════════════════════════════════════════════════════════

  Category distribution (clean dataset):
  ✔ caalamka          9,974   28.8%  ██████████████████████████████
  ✔ siyaasad          9,323   26.9%  ████████████████████████████
  ✔ ciyaaro           8,895   25.7%  ██████████████████████████
  ✔ amni              6,457   18.6%  ███████████████████

  Source distribution (clean dataset):
  goobjoog                12,564   36.3%  ██████████████████████████████
  caasimada                8,749   25.3%  ████████████████████
  kooxda                   7,163   20.7%  █████████████████
  mustaqbalmedia           4,516   13.0%  ██████████
  laacibnet                1,048    3.0%  ██
  bbc_somali                 235    0.7%  
  puntlandpost               218    0.6%  
  kubadlive                  111    0.3%  
  sonna                       25    0.1%  
  wararka24                   13  

## Cell 12 — Save Cleaned Files

In [19]:
"""
SAVE CLEANED FILES
==============================
Saves the cleaned dataset in two ways:

1. Per-category files (e.g. siyaasad_cleaned.csv)
   Useful when training category-specific models
   or inspecting one category at a time.

2. Combined file (combined_cleaned.csv)
   All categories in one file — useful for multi-class training.

Final column structure:
  site          — which website the article came from
  category      — siyaasad / amni / caalamka / ciyaaro
  url           — original article URL
  headline      — cleaned headline
  body          — cleaned body text
  scraped_at    — when it was scraped

Also saves suspicious_articles_full.csv with all flagged rows
in case you decide to include them after manual review.
"""

# Define final column order
OUTPUT_COLS = ['site', 'category', 'url', 'headline_clean', 'body_clean', 'scraped_at']
final_cols  = [c for c in OUTPUT_COLS if c in clean_df.columns]

save_df = clean_df[final_cols].copy()
save_df = save_df.rename(columns={
    'headline_clean': 'headline',
    'body_clean':     'body'
})

# ── Per-category files ────────────────────────────────────────────────────────
print('Saving per-category files...')
saved_counts = {}
for cat in sorted(save_df['category'].dropna().unique()):
    cat_df   = save_df[save_df['category'] == cat]
    out_path = CLEANED_DIR / f'{cat}_cleaned.csv'
    cat_df.to_csv(out_path, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compatibility
    saved_counts[cat] = len(cat_df)
    print(f'  ✔ {cat:<15} → {out_path.name}  ({len(cat_df):,} rows)')

# ── Combined file ─────────────────────────────────────────────────────────────
combined_path = CLEANED_DIR / 'combined_cleaned.csv'
save_df.to_csv(combined_path, index=False, encoding='utf-8-sig')
print(f'\n  ✔ Combined → combined_cleaned.csv ({len(save_df):,} rows total)')

# ── Suspicious articles (full columns, for review) ────────────────────────────
if len(suspicious_df) > 0:
    susp_save = suspicious_df[final_cols + ['_flag_str']].copy()
    susp_save = susp_save.rename(columns={'headline_clean':'headline','body_clean':'body'})
    susp_save.to_csv(REVIEW_DIR / 'suspicious_articles_full.csv', index=False, encoding='utf-8-sig')
    print(f'  ✔ Suspicious → review/suspicious_articles_full.csv ({len(suspicious_df):,} rows)')

print(f'\n  All files saved to: {CLEANED_DIR.resolve()}')

Saving per-category files...
  ✔ amni            → amni_cleaned.csv  (6,457 rows)
  ✔ caalamka        → caalamka_cleaned.csv  (9,974 rows)
  ✔ ciyaaro         → ciyaaro_cleaned.csv  (8,895 rows)
  ✔ siyaasad        → siyaasad_cleaned.csv  (9,323 rows)

  ✔ Combined → combined_cleaned.csv (34,649 rows total)
  ✔ Suspicious → review/suspicious_articles_full.csv (133 rows)

  All files saved to: E:\somali_cleaner\data\cleaned


In [22]:
# import importlib, scraper, pipeline
# importlib.reload(scraper); importlib.reload(pipeline)

import pandas as pd, re
df = pd.read_csv('data/raw/bbc_somali__ciyaaro.csv', dtype=str)

# Apply fix
df['body_clean'] = df['content'].apply(remove_bbc_image_captions)

# Check nothing remains
still_has = df['body_clean'].str.contains('Xigashada Sawirka', na=False).sum()
print(f'Articles still containing caption: {still_has} / {len(df)}')  # should be 0

# Before / after word count
before = df['content'].str.split().str.len().mean()
after  = df['body_clean'].str.split().str.len().mean()
print(f'Avg words before: {before:.1f}')
print(f'Avg words after : {after:.1f}')
print(f'Avg removed     : {before - after:.1f} words per article')

# Show 3 samples
for i in [1,2,3,4,5,6,7,8,9,10]:
    print(f'\n--- Article {i+1} ---')
    print(f'BEFORE: {df["content"].iloc[i][:150]}')
    print(f'AFTER : {df["body_clean"].iloc[i][:150]}')

Articles still containing caption: 0 / 245
Avg words before: 662.6
Avg words after : 646.4
Avg removed     : 16.2 words per article

--- Article 2 ---
BEFORE: Xigashada Sawirka, Getty Images Kooxda Real Madrid ayaa toddobaadkan gashay xaalad qalalaase weyn ah, xilli ay ahayd in dhammaan diiradda lagu saaro k
AFTER : Kooxda Real Madrid ayaa toddobaadkan gashay xaalad qalalaase weyn ah, xilli ay ahayd in dhammaan diiradda lagu saaro kulanka adag ee El Clasico ee ay 

--- Article 3 ---
BEFORE: Xigashada Sawirka, Getty Images Iyada oo ay soo dhowaatay isku-aadka Koobka Adduunka, dalal badan, gaar ahaan kuwa ka qayb galaya koobka ayaa bilaabay
AFTER : Iyada oo ay soo dhowaatay isku-aadka Koobka Adduunka, dalal badan, gaar ahaan kuwa ka qayb galaya koobka ayaa bilaabay qorshahooda inay dhowaan safraa

--- Article 4 ---
BEFORE: Xigashada Sawirka, Getty Images Arsenal waligeed kuma guuleysan Champions League iyo Europa League. Hadda waxay soo gaartay Semifinalka, waxayna barba
AFTER : Arsenal 

## Cell 13 — Build Model-Ready Dataset

In [ ]:
"""
BUILD MODEL-READY DATASET
=====================================
Prepares the final format for NLP headline generation fine-tuning.

Format:
  input_text  = '[CATEGORY] ' + cleaned body (truncated to MAX_BODY_WORDS)
  target_text = cleaned headline

Why include the category tag in input_text?
  The transformer model can learn category-aware generation.
  E.g. siyaasad articles use more political vocabulary,
  ciyaaro articles use sports terminology.
  Adding [SIYAASAD] as a prefix helps the model adapt its style.

Why truncate the body?
  Most transformers have a 512 or 1024 token context limit.
  Average Somali article is 200-400 words, so 512 words is safe.
  Very long articles (BBC: avg 662 words) are cleanly truncated.

Saved as: combined_model_ready.csv
  Columns: site, category, url, input_text, target_text
"""

MAX_BODY_WORDS = 512   # adjust if your model supports longer context

def build_input_text(row) -> str:
    """
    Builds the model input string.
    Format: '[CATEGORY] <first N words of cleaned body>'

    The category tag is uppercased and wrapped in square brackets
    so it's easy to add as a special token in the tokenizer later.
    """
    category_tag = f"[{str(row['category']).upper()}]"
    body_words   = str(row['body']).split()
    truncated    = ' '.join(body_words[:MAX_BODY_WORDS])
    return f"{category_tag} {truncated}"


model_df = save_df.copy()
model_df['input_text']  = model_df.apply(build_input_text, axis=1)
model_df['target_text'] = model_df['headline']  # the headline IS the target

# Save
model_cols = ['site', 'category', 'url', 'input_text', 'target_text']
model_path = CLEANED_DIR / 'combined_model_ready.csv'
model_df[model_cols].to_csv(model_path, index=False, encoding='utf-8-sig')

print(f'  ✔ combined_model_ready.csv ({len(model_df):,} rows)')

# Show sample
print('\n  ── Sample model-ready row ──────────────────────────────────')
sample = model_df[model_df['category'] == 'siyaasad'].sample(1, random_state=3).iloc[0]
print(f'  TARGET : {sample["target_text"]}')
print(f'  INPUT  : {sample["input_text"][:300]}...')

# Per-category model files
print('\n  Saving per-category model files...')
for cat in sorted(model_df['category'].dropna().unique()):
    cat_model = model_df[model_df['category'] == cat][model_cols]
    out = CLEANED_DIR / f'{cat}_model_ready.csv'
    cat_model.to_csv(out, index=False, encoding='utf-8-sig')
    print(f'  ✔ {cat:<15} → {out.name}  ({len(cat_model):,} rows)')

## Cell 14 — Final Summary Report

In [23]:
"""
CELL 14 — FINAL SUMMARY REPORT
================================
Prints a complete summary of what was done and saves it to a CSV.

Includes:
  - Article count at each pipeline stage
  - Final category and site distribution
  - Headline and body quality statistics
  - List of all output files with sizes

Also saves cleaning_summary_YYYY-MM-DD.csv to data/reports/
so you have a permanent record of what this cleaning run produced.
"""

import csv as csv_mod

print('═' * 65)
print('  CLEANING PIPELINE — FINAL SUMMARY REPORT')
print(f'  Run date: {TODAY}')
print('═' * 65)

# ── Pipeline stages ───────────────────────────────────────────────────────────
print(f'\n  {"Stage":<42} {"Rows":>8}')
print(f'  {"─" * 52}')
stages = [
    ('Raw articles loaded',              TOTAL_RAW),
    ('After removing bad article types', len(df) + len(bad_df)),   # approx
    ('After missing value removal',      None),
    ('After deduplication',              None),
    ('Flagged as suspicious (review)',   len(suspicious_df)),
    ('Final clean articles',             len(save_df)),
]
report_rows = []
for label, count in stages:
    val = f'{count:,}' if count is not None else 'see cells above'
    print(f'  {label:<42} {val:>8}')
    report_rows.append({'stage': label, 'count': count})

# ── Category distribution ─────────────────────────────────────────────────────
print(f'\n  {"Category":<18} {"Articles":>9} {"% of total":>12}')
print(f'  {"─" * 42}')
for cat, n in save_df['category'].value_counts().items():
    pct = n / len(save_df) * 100
    print(f'  {cat:<18} {n:>9,} {pct:>11.1f}%')

# ── Source distribution ───────────────────────────────────────────────────────
print(f'\n  {"Source":<22} {"Articles":>9}')
print(f'  {"─" * 33}')
for site, n in save_df['site'].value_counts().items():
    print(f'  {site:<22} {n:>9,}')

# ── Quality stats ─────────────────────────────────────────────────────────────
hl_lens  = save_df['headline'].str.len()
bd_words = save_df['body'].str.split().str.len()
print(f'\n  Headline chars  — avg: {hl_lens.mean():.0f}  min: {hl_lens.min()}  max: {hl_lens.max()}')
print(f'  Body words      — avg: {bd_words.mean():.0f}  min: {bd_words.min()}  max: {bd_words.max()}')

# ── Output files ──────────────────────────────────────────────────────────────
print('\n  Output files:')
all_outputs = (
    list(CLEANED_DIR.glob('*.csv')) +
    list(REVIEW_DIR.glob('*.csv')) +
    list(REPORTS_DIR.glob('*.csv'))
)
for fp in sorted(all_outputs):
    size_kb = fp.stat().st_size // 1024
    rel     = fp.relative_to(BASE_DIR)
    print(f'  {str(rel):<55} {size_kb:>6} KB')

# ── Save CSV report ───────────────────────────────────────────────────────────
report_path = REPORTS_DIR / f'cleaning_summary_{TODAY}.csv'
with open(report_path, 'w', newline='', encoding='utf-8') as f:
    w = csv_mod.DictWriter(f, fieldnames=['stage', 'count'])
    w.writeheader()
    w.writerows(report_rows)

print(f'\n  Report saved → {report_path}')
print('\n' + '═' * 65)
print('  ✅ Cleaning pipeline complete.')
print('═' * 65)

═════════════════════════════════════════════════════════════════
  CLEANING PIPELINE — FINAL SUMMARY REPORT
  Run date: 2026-05-22
═════════════════════════════════════════════════════════════════

  Stage                                          Rows
  ────────────────────────────────────────────────────
  Raw articles loaded                          35,797
  After removing bad article types                290
  After missing value removal                see cells above
  After deduplication                        see cells above
  Flagged as suspicious (review)                  133
  Final clean articles                         34,649

  Category            Articles   % of total
  ──────────────────────────────────────────
  caalamka               9,974        28.8%
  siyaasad               9,323        26.9%
  ciyaaro                8,895        25.7%
  amni                   6,457        18.6%

  Source                  Articles
  ─────────────────────────────────
  goobjoog      